In [1]:
!nvidia-smi
!nvcc --version

Fri Jun 19 08:11:46 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   36C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
%%writefile mylib.cu
#include <cuda_runtime.h>
#include <iostream>

extern "C" __global__
void addKernel(int* c, const int* a, const int* b, int size) {
    int i = blockIdx.x * blockDim.x + threadIdx.x;
    if (i < size) c[i] = a[i] + b[i];
}

extern "C" void addVectors(int* c, const int* a, const int* b, int size) {
    int *d_a, *d_b, *d_c;
    size_t bytes = size * sizeof(int);
    cudaMalloc(&d_a, bytes);
    cudaMalloc(&d_b, bytes);
    cudaMalloc(&d_c, bytes);
    cudaMemcpy(d_a, a, bytes, cudaMemcpyHostToDevice);
    cudaMemcpy(d_b, b, bytes, cudaMemcpyHostToDevice);
    int threadsPerBlock = 256;
    int blocks = (size + threadsPerBlock - 1) / threadsPerBlock;
    addKernel<<<blocks, threadsPerBlock>>>(d_c, d_a, d_b, size);
    cudaDeviceSynchronize();
    cudaMemcpy(c, d_c, bytes, cudaMemcpyDeviceToHost);
    cudaFree(d_a); cudaFree(d_b); cudaFree(d_c);
}

Writing mylib.cu


In [3]:
%%writefile mylib2d.cu
#include <cuda_runtime.h>
#include <iostream>

__global__
void addKernel2D(int* c, const int* a, const int* b, int rows, int cols) {
    int row = blockIdx.y * blockDim.y + threadIdx.y;
    int col = blockIdx.x * blockDim.x + threadIdx.x;
    if (row < rows && col < cols) {
        int idx = row * cols + col;
        c[idx] = a[idx] + b[idx];
    }
}

extern "C"
void addArrays2D(int* c, const int* a, const int* b, int rows, int cols) {
    int total = rows * cols;
    size_t bytes = total * sizeof(int);
    int *d_a = nullptr, *d_b = nullptr, *d_c = nullptr;
    cudaError_t err;
    err = cudaMalloc(&d_a, bytes);
    if (err == cudaSuccess) err = cudaMalloc(&d_b, bytes);
    if (err == cudaSuccess) err = cudaMalloc(&d_c, bytes);
    if (err != cudaSuccess) { std::cerr << "cudaMalloc failed" << std::endl; return; }
    cudaMemcpy(d_a, a, bytes, cudaMemcpyHostToDevice);
    cudaMemcpy(d_b, b, bytes, cudaMemcpyHostToDevice);
    dim3 threads(16, 16);
    dim3 blocks((cols + threads.x - 1) / threads.x, (rows + threads.y - 1) / threads.y);
    addKernel2D<<<blocks, threads>>>(d_c, d_a, d_b, rows, cols);
    err = cudaDeviceSynchronize();
    if (err != cudaSuccess) { std::cerr << "Kernel error" << std::endl; return; }
    cudaMemcpy(c, d_c, bytes, cudaMemcpyDeviceToHost);
    cudaFree(d_a); cudaFree(d_b); cudaFree(d_c);
}

Writing mylib2d.cu


In [4]:
%%writefile main.cpp
#include <dlfcn.h>
#include <iostream>

typedef void (*AddVectorsFunc)(int*, const int*, const int*, int);

int main() {
    void* handle = dlopen("./libmylib.so", RTLD_LAZY);
    if (!handle) { std::cerr << "Failed to load: " << dlerror() << std::endl; return 1; }
    AddVectorsFunc addVectors = (AddVectorsFunc)dlsym(handle, "addVectors");
    if (!addVectors) { std::cerr << "Function not found: " << dlerror() << std::endl; return 1; }
    int a[5] = {1,2,3,4,5};
    int b[5] = {10,20,30,40,50};
    int c[5];
    addVectors(c, a, b, 5);
    for (int i : c) std::cout << i << " ";
    std::cout << std::endl;
    dlclose(handle);
    return 0;
}

Writing main.cpp


In [5]:
%%writefile main2d.cpp
#include <dlfcn.h>
#include <iostream>

#define ROWS 3
#define COLS 4

typedef void (*AddArrays2DFunc)(int*, const int*, const int*, int, int);

int main() {
    void* handle = dlopen("./libmylib2d.so", RTLD_LAZY);
    if (!handle) {
        std::cerr << "Failed to load: " << dlerror() << std::endl;
        return 1;
    }

    AddArrays2DFunc addArrays2D =
        (AddArrays2DFunc)dlsym(handle, "addArrays2D");

    if (!addArrays2D) {
        std::cerr << "Function not found: " << dlerror() << std::endl;
        return 1;
    }

    int a[ROWS][COLS] = {
        {1,2,3,4},
        {5,6,7,8},
        {9,10,11,12}
    };

    int b[ROWS][COLS] = {
        {10,20,30,40},
        {50,60,70,80},
        {90,100,110,120}
    };

    int c[ROWS][COLS];

    // Show input array A
    std::cout << "Input Array A:" << std::endl;
    for(int r = 0; r < ROWS; r++) {
        for(int col = 0; col < COLS; col++) {
            std::cout << a[r][col] << "\t";
        }
        std::cout << std::endl;
    }

    // Show input array B
    std::cout << "\nInput Array B:" << std::endl;
    for(int r = 0; r < ROWS; r++) {
        for(int col = 0; col < COLS; col++) {
            std::cout << b[r][col] << "\t";
        }
        std::cout << std::endl;
    }

    addArrays2D(&c[0][0], &a[0][0], &b[0][0], ROWS, COLS);

    // Show output array C
    std::cout << "\nOutput Array C (A + B):" << std::endl;
    for(int r = 0; r < ROWS; r++) {
        for(int col = 0; col < COLS; col++) {
            std::cout << c[r][col] << "\t";
        }
        std::cout << std::endl;
    }

    dlclose(handle);
    return 0;
}

Writing main2d.cpp


In [9]:
!nvcc -Xcompiler -fPIC -shared -o libmylib.so mylib.cu
!nvcc -Xcompiler -fPIC -shared -o libmylib2d.so mylib2d.cu
!g++ -o main main.cpp -I. -L. -lmylib -Wl,-rpath,. -ldl
!g++ -o main2d main2d.cpp -I. -L. -lmylib2d -Wl,-rpath,. -ldl
!ls -lh libmylib.so libmylib2d.so main main2d

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).
nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).
-rwxr-xr-x 1 root root 982K Jun 19 08:15 libmylib2d.so
-rwxr-xr-x 1 root root 981K Jun 19 08:15 libmylib.so
-rwxr-xr-x 1 root root  17K Jun 19 08:15 main
-rwxr-xr-x 1 root root  17K Jun 19 08:15 main2d


In [10]:
!./main

11 22 33 44 55 


In [11]:
!./main2d

Input Array A:
1	2	3	4	
5	6	7	8	
9	10	11	12	

Input Array B:
10	20	30	40	
50	60	70	80	
90	100	110	120	

Output Array C (A + B):
11	22	33	44	
55	66	77	88	
99	110	121	132	
